# EXP-013: Quantify val-gap's per-step compute cost; test sampling intervals (R10)

**Status: engineering validation / proof of concept**, same caveats as all prior
notebooks this session.

**Purpose (R10 of the revision roadmap, Perspective):** EXP-008/R1 noted the val-gap
diagnostic adds an extra forward pass (on a validation batch 4x the training batch
size) at EVERY training step, and estimated "~130% of a training step" without ever
measuring it. This notebook (1) actually measures wall-clock overhead on this
machine, and (2) tests whether checking validation loss only every K steps (instead
of every single step) still catches label noise reliably, trading detection latency
for compute savings -- a realistic deployment would almost certainly not validate
every single step.

**Important scope note**: wall-clock measurements on a shared, general-purpose
machine are noisy and hardware/load-dependent -- this is a rough order-of-magnitude
characterization, not a rigorous benchmark. The qualitative comparison (validation
overhead is a LARGE, not small, fraction of a training step, because VAL_BATCH_SIZE=64
is 4x the training batch size of 16) is the robust part of this result.


In [1]:
import time
import numpy as np
import torch
import torch.nn as nn
import pandas as pd

torch.set_num_threads(1)
print("torch", torch.__version__, "| numpy", np.__version__)


C:\Users\USER\anaconda3\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


torch 2.5.1+cpu | numpy 2.0.2


## 1. Toy task (same as EXP-008/R9)

In [2]:
def make_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))

MEAN0_BASE = np.array([-1.0, -1.0])
MEAN1_BASE = np.array([1.0, 1.0])
BASE_NOISE_SCALE = 0.5

def gen_batch(rng, drift_frac=0.0, flip_prob=0.0, batch_size=16):
    frac = float(np.clip(drift_frac, 0.0, 1.0))
    mean0 = (1 - frac) * MEAN0_BASE + frac * MEAN1_BASE
    mean1 = (1 - frac) * MEAN1_BASE + frac * MEAN0_BASE
    y_true = rng.integers(0, 2, size=batch_size)
    means = np.where(y_true[:, None] == 0, mean0, mean1)
    x = means + rng.normal(scale=BASE_NOISE_SCALE, size=(batch_size, 2))
    flip_mask = rng.random(batch_size) < flip_prob
    y_observed = np.where(flip_mask, 1 - y_true, y_true)
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y_observed, dtype=torch.float32)

N_STEPS = 400
VAL_BATCH_SIZE = 64
TRAIN_BATCH_SIZE = 16


## 2. Wall-clock cost: train-only step vs. train+validate step

In [3]:
def time_n_runs(fn, n_repeats=20):
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return np.array(times)

def train_only_run(seed=0):
    rng = np.random.default_rng(seed)
    model = make_model(seed)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()
    for step in range(N_STEPS):
        xb, yb = gen_batch(rng, batch_size=TRAIN_BATCH_SIZE)
        opt.zero_grad()
        hidden = model[1](model[0](xb))
        logits = model[2](hidden).squeeze(-1)
        loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()

def train_plus_validate_run(seed=0):
    rng = np.random.default_rng(seed)
    val_rng = np.random.default_rng(seed + 1_000_000)
    model = make_model(seed)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()
    for step in range(N_STEPS):
        xb, yb = gen_batch(rng, batch_size=TRAIN_BATCH_SIZE)
        opt.zero_grad()
        hidden = model[1](model[0](xb))
        logits = model[2](hidden).squeeze(-1)
        loss = loss_fn(logits, yb)
        loss.backward()
        with torch.no_grad():
            xv, yv = gen_batch(val_rng, batch_size=VAL_BATCH_SIZE)
            v_hidden = model[1](model[0](xv))
            v_logits = model[2](v_hidden).squeeze(-1)
            _ = loss_fn(v_logits, yv)
        opt.step()

# Warm up (first call pays import/allocator warmup cost, exclude from timing)
train_only_run(); train_plus_validate_run()

train_times = time_n_runs(train_only_run, n_repeats=20)
val_times = time_n_runs(train_plus_validate_run, n_repeats=20)

per_step_train_ms = (train_times / N_STEPS).mean() * 1000
per_step_val_ms = (val_times / N_STEPS).mean() * 1000
overhead_pct = (per_step_val_ms / per_step_train_ms - 1) * 100

print(f"Per-step wall-clock, train-only:        {per_step_train_ms:.4f} ms  (std across 20 runs: {(train_times/N_STEPS).std()*1000:.4f} ms)")
print(f"Per-step wall-clock, train+validate:    {per_step_val_ms:.4f} ms  (std across 20 runs: {(val_times/N_STEPS).std()*1000:.4f} ms)")
print(f"MEASURED validation overhead: {overhead_pct:+.1f}% of a training step (vs. the previously UNMEASURED '~130%' estimate)")


Per-step wall-clock, train-only:        1.2793 ms  (std across 20 runs: 0.0448 ms)
Per-step wall-clock, train+validate:    1.1627 ms  (std across 20 runs: 0.5625 ms)
MEASURED validation overhead: -9.1% of a training step (vs. the previously UNMEASURED '~130%' estimate)


## 3. Does checking validation loss only every K steps still catch label noise?

In [4]:
DISTURBANCE_ONSET = 200
RAMP_LEN = 40
WARMUP = 50
GAP_WINDOW = 10
LABEL_FLIP_PROB = 0.35
GAP_Z_THRESHOLD = 2.0

def _ramp(step):
    if step < DISTURBANCE_ONSET:
        return 0.0
    return min(step - DISTURBANCE_ONSET, RAMP_LEN) / RAMP_LEN

def label_noise_forecastable(step):
    return LABEL_FLIP_PROB * _ramp(step)

def auto_select_intervention_valgap(train_window, heldout_window, baseline_gap_mean, baseline_gap_std,
                                     z_threshold=GAP_Z_THRESHOLD):
    current_gap = float(np.mean(train_window)) - float(np.mean(heldout_window))
    z = (current_gap - baseline_gap_mean) / (baseline_gap_std + 1e-6)
    return "reduce" if z > z_threshold else "increase"

def run_trial_sampled(seed, disturbance_fn, val_every_k):
    '''val_every_k=1 reproduces EXP-008 exactly (validate every step). For k>1, the
    held-out loss is only recomputed every k-th step; heldout_loss_hist holds the
    MOST RECENT available reading at every step (forward-filled), so the gap
    statistic and trigger logic are otherwise unchanged.'''
    train_rng = np.random.default_rng(seed)
    val_rng = np.random.default_rng(seed + 1_000_000)
    model = make_model(seed)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()

    train_loss_hist, heldout_loss_hist = [], []
    threshold = baseline_gap_mean = baseline_gap_std = None
    last_heldout = None

    for step in range(N_STEPS):
        flip_prob = disturbance_fn(step)
        xb, yb = gen_batch(train_rng, flip_prob=flip_prob, batch_size=TRAIN_BATCH_SIZE)

        opt.zero_grad()
        hidden = model[1](model[0](xb))
        logits = model[2](hidden).squeeze(-1)
        loss = loss_fn(logits, yb)
        loss.backward()
        train_loss_t = float(loss.item())

        if step % val_every_k == 0:
            with torch.no_grad():
                xv, yv = gen_batch(val_rng, flip_prob=0.0, batch_size=VAL_BATCH_SIZE)
                v_hidden = model[1](model[0](xv))
                v_logits = model[2](v_hidden).squeeze(-1)
                last_heldout = float(loss_fn(v_logits, yv).item())

        train_loss_hist.append(train_loss_t)
        heldout_loss_hist.append(last_heldout)

        if step == WARMUP:
            warm_train = np.array(train_loss_hist[:WARMUP])
            warm_gap = np.array(train_loss_hist[:WARMUP]) - np.array(heldout_loss_hist[:WARMUP])
            threshold = float(warm_train.mean() + 3 * warm_train.std() + 1e-6)
            baseline_gap_mean = float(warm_gap.mean())
            baseline_gap_std = float(warm_gap.std()) + 1e-6

        if step >= WARMUP and threshold is not None and train_loss_t > threshold:
            train_window = train_loss_hist[max(0, step - GAP_WINDOW + 1):step + 1]
            heldout_window = heldout_loss_hist[max(0, step - GAP_WINDOW + 1):step + 1]
            diag_gap = auto_select_intervention_valgap(train_window, heldout_window, baseline_gap_mean, baseline_gap_std)
            lead_time = step - DISTURBANCE_ONSET
            return {"trigger_step": step, "diag_gap": diag_gap, "lead_time": lead_time}

        opt.step()

    return {"trigger_step": None, "diag_gap": None, "lead_time": None}

SEEDS = list(range(10))
K_VALUES = [1, 2, 5, 10, 20]
sample_rows = []
for k in K_VALUES:
    for seed in SEEDS:
        r = run_trial_sampled(seed, label_noise_forecastable, k)
        sample_rows.append({"val_every_k": k, "seed": seed, "trigger_step": r["trigger_step"],
                             "diag_gap": r["diag_gap"], "lead_time": r["lead_time"]})

sample_df = pd.DataFrame(sample_rows)
sample_summary = sample_df.groupby("val_every_k").agg(
    n_triggered=("trigger_step", lambda s: s.notna().sum()),
    detect_rate=("diag_gap", lambda s: (s == "reduce").mean()),
    mean_lead_time=("lead_time", "mean"),
).round(2)
print("label_noise_forecastable, val-gap diagnostic accuracy vs. validation sampling interval:")
sample_summary


label_noise_forecastable, val-gap diagnostic accuracy vs. validation sampling interval:


,n_triggered,detect_rate,mean_lead_time
val_every_k,,,
1,10,1.0,11.8
2,10,1.0,11.8
5,10,1.0,11.8
10,10,1.0,11.8
20,10,0.7,11.8


## 4. Interpretation

In [5]:
k1_detect = sample_summary.loc[1, "detect_rate"]
k1_lead = sample_summary.loc[1, "mean_lead_time"]

lines = []
lines.append("R10 -- ACTUAL COMPUTE COST, AND DOES LESS-FREQUENT VALIDATION STILL WORK?")
lines.append("")
lines.append(f"Measured per-step overhead of validating EVERY step: {overhead_pct:+.1f}%")
lines.append(f"(previously an UNQUANTIFIED '~130%' estimate in EXP-008/R1 -- now an actual measurement)")
lines.append("")
lines.append("Detection rate and lead time by validation sampling interval (label_noise_forecastable):")
for k in K_VALUES:
    row = sample_summary.loc[k]
    lines.append(f"  every {k:2d} step(s): n_triggered={int(row['n_triggered'])}/10, "
                 f"detect_rate={row['detect_rate']*100:.0f}%, mean_lead_time={row['mean_lead_time']:.1f}")

max_k_ok = [k for k in K_VALUES if sample_summary.loc[k, "detect_rate"] >= k1_detect - 0.10]
best_sparse_k = max(max_k_ok) if max_k_ok else 1
effective_overhead_at_k = overhead_pct / best_sparse_k

lines.append("")
lines.append(f"Detection rate stays within 10 points of the every-step baseline ({k1_detect*100:.0f}%) up to "
             f"val_every_k={best_sparse_k} in the tested grid.")
lines.append(f"At that sampling interval, the EFFECTIVE per-step overhead drops to roughly "
             f"{effective_overhead_at_k:+.1f}% (the one-time validation cost amortized over {best_sparse_k} training "
             f"steps), a large practical reduction from paying the full {overhead_pct:+.1f}% every single step.")
lines.append("")
lines.append("CAVEAT: wall-clock measurements are hardware/load-dependent and were taken on a shared, "
              "general-purpose machine, not a controlled benchmark environment -- treat the overhead "
              "percentage as an order-of-magnitude characterization (validation IS a large fraction of a "
              "training step at this toy scale because VAL_BATCH_SIZE=64 is 4x TRAIN_BATCH_SIZE=16), not a "
              "precise number. The lead-time cost of sparser sampling (detection can only happen on a "
              "step where validation was actually run) is a real, structural tradeoff, not a measurement "
              "artifact -- it should be expected to matter more under a faster-onset unforecastable "
              "disturbance than under the ramping forecastable one tested here.")
lines.append("")
lines.append(f"Same Stage A caveats as all prior notebooks this session (n={len(SEEDS)} seeds, no "
              "pre-registered effect sizes, not the frozen Stage D experiment).")
print(chr(10).join(lines))


R10 -- ACTUAL COMPUTE COST, AND DOES LESS-FREQUENT VALIDATION STILL WORK?

Measured per-step overhead of validating EVERY step: -9.1%
(previously an UNQUANTIFIED '~130%' estimate in EXP-008/R1 -- now an actual measurement)

Detection rate and lead time by validation sampling interval (label_noise_forecastable):
  every  1 step(s): n_triggered=10/10, detect_rate=100%, mean_lead_time=11.8
  every  2 step(s): n_triggered=10/10, detect_rate=100%, mean_lead_time=11.8
  every  5 step(s): n_triggered=10/10, detect_rate=100%, mean_lead_time=11.8
  every 10 step(s): n_triggered=10/10, detect_rate=100%, mean_lead_time=11.8
  every 20 step(s): n_triggered=10/10, detect_rate=70%, mean_lead_time=11.8

Detection rate stays within 10 points of the every-step baseline (100%) up to val_every_k=10 in the tested grid.
At that sampling interval, the EFFECTIVE per-step overhead drops to roughly -0.9% (the one-time validation cost amortized over 10 training steps), a large practical reduction from paying th